# Genie Space Export (Databricks notebook)
Pulls one Genie Space and saves it as YAML in `genie-export/`.

Host and token come from the notebook context, so only `SPACE_ID` is needed.

In [ ]:
import json
import re
from pathlib import Path
from typing import Any, Dict

import requests
import yaml

In [ ]:
# ---- Config ----
SPACE_ID = ""   # Genie Space id

# Host + token + user from the notebook context
context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
hostname = context.apiUrl().get()
token = context.apiToken().get()
user = context.userName().get()
header = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Output goes OUTSIDE the cloned Git folder so it never shows up in git status/commits.
# Option A (default): your personal workspace folder, persists across cluster restarts
EXPORT_DIR = Path(f"/Workspace/Users/{user}/genie-export")
# Option B: driver-local temp disk (not tracked, lost on cluster restart)
# EXPORT_DIR = Path("/tmp/genie-export")
# Option C: a Unity Catalog volume
# EXPORT_DIR = Path("/Volumes/<catalog>/<schema>/<volume>/genie-export")

In [ ]:
def slugify(value: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "_", value.strip().lower())
    return slug.strip("_") or "untitled"


def fetch_space(space_id: str) -> dict:
    print(f"Fetching Genie Space {space_id} ...")
    resp = requests.get(
        f"{hostname.rstrip('/')}/api/2.0/genie/spaces/{space_id}",
        headers=header,
        params={"include_serialized_space": "true"},
        timeout=60,
    )
    if not resp.ok:
        try:
            body: Any = resp.json()
        except ValueError:
            body = resp.text
        raise RuntimeError(f"fetch_space failed: HTTP {resp.status_code} - {body}")
    return resp.json()

In [ ]:
if not SPACE_ID.strip():
    raise ValueError("Set SPACE_ID in the config cell.")

raw_space = fetch_space(SPACE_ID.strip())
inner = json.loads(raw_space.get("serialized_space") or "{}")

out: Dict[str, Any] = {
    "title": raw_space.get("title"),
    "description": raw_space.get("description"),
}
if raw_space.get("warehouse_id"):
    out["warehouse_id"] = raw_space["warehouse_id"]
out["config"] = inner.get("config", {})
out["data_sources"] = inner.get("data_sources", {})
out["instructions"] = inner.get("instructions", {})
out["benchmarks"] = inner.get("benchmarks", {})

cwd = Path.cwd().resolve()
if cwd == EXPORT_DIR.resolve() or cwd in EXPORT_DIR.resolve().parents:
    raise RuntimeError(f"EXPORT_DIR {EXPORT_DIR} is inside the notebook's folder ({cwd}), "
                       "which is the Git repo. Choose a path outside it.")

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
out_path = EXPORT_DIR / f"{slugify(raw_space.get('title') or SPACE_ID)}.yaml"
with out_path.open("w", encoding="utf-8") as handle:
    yaml.dump(out, handle, sort_keys=False, default_flow_style=False, allow_unicode=True)

print(f"Saved: {out_path.resolve()}")